# Sesión 04 — Medallion Architecture: Capa Bronze
**Curso:** Databricks Data Engineer Associate  
**Runtime mínimo:** Databricks 13.3 LTS  
**Catálogo:** `dbassociate` | **Esquema:** `default`  
**Volume:** `/Volumes/dbassociate/default/vol_landing/`

## Objetivos del laboratorio
1. Diseñar e implementar una tabla Bronze con columnas técnicas de auditoría
2. Ingestar datos de inventario farmacéutico en modo append con idempotencia
3. Implementar particionamiento por fecha de ingesta
4. Construir una tabla `ingestion_log` para observabilidad operativa
5. Ejecutar OPTIMIZE y verificar el estado del transaction log

## Archivos requeridos
Subir al Volume antes de ejecutar:
- `inventario_batch01.csv` → `/Volumes/dbassociate/default/vol_landing/session04/`
- `inventario_batch02.csv` → `/Volumes/dbassociate/default/vol_landing/session04/`
- `almacenes_referencia.csv` → `/Volumes/dbassociate/default/vol_landing/session04/`


## Celda 1 — Configuración global del laboratorio

In [0]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, DateType, TimestampType
)

# Constantes del laboratorio
CATALOG      = "dbassociate"
SCHEMA       = "default"
VOLUME_PATH  = f"/Volumes/{CATALOG}/{SCHEMA}/vol_landing/session04"
TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_inventario_farmaceutico"
TABLE_LOG    = f"{CATALOG}.{SCHEMA}.bronze_ingestion_log"
SOURCE_SYSTEM = "erp_farmacia_v2"

# batch_id: identifica de forma única esta ejecución del pipeline
# En producción se usaría dbutils.jobs.taskValues o el run_id del workflow
BATCH_ID = str(uuid.uuid4())

print(f"CATALOG      : {CATALOG}")
print(f"SCHEMA       : {SCHEMA}")
print(f"VOLUME_PATH  : {VOLUME_PATH}")
print(f"TABLE_BRONZE : {TABLE_BRONZE}")
print(f"TABLE_LOG    : {TABLE_LOG}")
print(f"BATCH_ID     : {BATCH_ID}")
print(f"SOURCE_SYSTEM: {SOURCE_SYSTEM}")


## Celda 2 — Verificar que los archivos fuente están disponibles

Antes de ejecutar el pipeline, verificamos que los archivos estén en el Volume.  
Si esta celda falla, revisar que los CSV fueron subidos correctamente.


In [0]:
import os

archivos_esperados = [
    f"{VOLUME_PATH}/inventario_batch01.csv",
    f"{VOLUME_PATH}/inventario_batch02.csv",
    f"{VOLUME_PATH}/almacenes_referencia.csv",
]

todos_presentes = True
for ruta in archivos_esperados:
    existe = os.path.exists(ruta)
    estado = "OK" if existe else "FALTA"
    print(f"[{estado}] {ruta}")
    if not existe:
        todos_presentes = False

if not todos_presentes:
    raise FileNotFoundError(
        "Uno o más archivos fuente no están disponibles. "
        "Subir los CSV al Volume antes de continuar."
    )

print("\nTodos los archivos fuente verificados.")


## Celda 3 — Definir el esquema explícito de la fuente

En Bronze se recomienda definir un esquema explícito para detectar cambios  
no esperados en el origen. La inferencia automática puede enmascarar problemas.

> **Antipatrón:** usar `inferSchema=True` sin validación en Bronze de producción.  
> Si el origen cambia un tipo de dato, la inferencia puede silenciosamente  
> alterar la semántica de los datos.


In [0]:
# Esquema del archivo fuente (contrato con el sistema de origen)
schema_fuente = StructType([
    StructField("producto_id",      StringType(),  nullable=False),
    StructField("sku",              StringType(),  nullable=True),
    StructField("nombre_producto",  StringType(),  nullable=True),
    StructField("categoria",        StringType(),  nullable=True),
    StructField("laboratorio",      StringType(),  nullable=True),
    StructField("unidades_stock",   IntegerType(), nullable=True),
    StructField("precio_unitario",  DoubleType(),  nullable=True),
    StructField("fecha_vencimiento",DateType(),    nullable=True),
    StructField("lote",             StringType(),  nullable=True),
    StructField("almacen_id",       StringType(),  nullable=True),
])

print("Esquema de la fuente definido:")
for campo in schema_fuente.fields:
    nullable_str = "nullable" if campo.nullable else "NOT NULL"
    print(f"  {campo.name:<25} {str(campo.dataType):<20} {nullable_str}")


## Celda 4 — Función de ingesta con columnas técnicas de auditoría

Esta función encapsula la lógica de ingesta Bronze:
- Lee el archivo con esquema explícito
- Agrega las columnas técnicas de auditoría
- Escribe en Delta con modo `append` y particionamiento por fecha de ingesta
- Retorna métricas para el ingestion_log

> **Nota sobre `col("_metadata.file_name")`:**  
> Esta es la forma correcta en DBR 13.3+. La función `input_file_name()`  
> está deprecada y puede retornar resultados incorrectos en operaciones  
> con optimizaciones de Spark activadas.


In [0]:
def ingestar_batch_bronze(ruta_archivo: str, batch_id: str) -> dict:
    """
    Ingesta un archivo CSV al Bronze layer con columnas técnicas.
    
    Parámetros:
        ruta_archivo (str): Ruta completa al archivo en el Volume
        batch_id (str): Identificador único de esta ejecución
    
    Retorna:
        dict: Métricas de la ingesta para el log de control
    """
    inicio = datetime.now()
    
    # Leer con esquema explícito y captura de metadata
    df_raw = (
        spark.read
        .schema(schema_fuente)
        .option("header", "true")
        .option("dateFormat", "yyyy-MM-dd")
        # mode FAILFAST: falla si un registro no cumple el esquema
        # Alternativa PERMISSIVE (default): registros malformados → nulls
        # Alternativa DROPMALFORMED: descarta registros malformados
        .option("mode", "FAILFAST")
        .csv(ruta_archivo)
    )
    
    registros_leidos = df_raw.count()
    
    # Agregar columnas técnicas de auditoría Bronze
    # ingestion_timestamp: momento de procesamiento (no del archivo fuente)
    # source_file: nombre del archivo de origen para trazabilidad
    # source_system: sistema de origen del dato
    # batch_id: identificador de la ejecución del pipeline
    # ingestion_date: columna de partición (derivada del timestamp)
    df_bronze = df_raw.withColumns({
        "ingestion_timestamp": F.current_timestamp(),
        "source_file":         F.lit(ruta_archivo.split("/")[-1]),
        "source_system":       F.lit(SOURCE_SYSTEM),
        "batch_id":            F.lit(batch_id),
        "ingestion_date":      F.to_date(F.current_timestamp()),
    })
    
    # Escribir en Delta con particionamiento por fecha de ingesta
    # Nota: particionamos por ingestion_date (fecha técnica), NO por
    # fecha_vencimiento ni ninguna columna de negocio
    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .partitionBy("ingestion_date")
        .saveAsTable(TABLE_BRONZE)
    )
    
    fin = datetime.now()
    duracion_seg = (fin - inicio).total_seconds()
    
    return {
        "batch_id":         batch_id,
        "archivo":          ruta_archivo.split("/")[-1],
        "source_system":    SOURCE_SYSTEM,
        "registros_leidos": registros_leidos,
        "estado":           "COMPLETADO",
        "inicio":           inicio,
        "fin":              fin,
        "duracion_seg":     duracion_seg,
        "tabla_destino":    TABLE_BRONZE,
    }

print("Función ingestar_batch_bronze definida.")


## Celda 5 — Crear la tabla Bronze (primera ejecución)

Creamos la tabla Delta vacía con el esquema completo incluyendo columnas técnicas  
y particionamiento definido.

> **Por qué crear la tabla explícitamente antes de escribir:**  
> Permite agregar propiedades Delta importantes desde el inicio,  
> como habilitar Change Data Feed (`delta.enableChangeDataFeed`).  
> Si dejamos que el primer `write` cree la tabla, estas propiedades  
> deben agregarse después con `ALTER TABLE`, lo que genera más deuda técnica.


In [0]:
# Crear la tabla Bronze si no existe
# Incluimos delta.enableChangeDataFeed desde el inicio
# (se usará en sesión 07 para alimentar Silver de forma incremental)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_BRONZE} (
        -- Columnas de negocio (origen)
        producto_id       STRING        NOT NULL,
        sku               STRING,
        nombre_producto   STRING,
        categoria         STRING,
        laboratorio       STRING,
        unidades_stock    INT,
        precio_unitario   DOUBLE,
        fecha_vencimiento DATE,
        lote              STRING,
        almacen_id        STRING,
        -- Columnas técnicas de auditoría Bronze
        ingestion_timestamp TIMESTAMP,
        source_file         STRING,
        source_system       STRING,
        batch_id            STRING,
        ingestion_date      DATE
    )
    USING DELTA
    PARTITIONED BY (ingestion_date)
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'delta.minReaderVersion'     = '1',
        'delta.minWriterVersion'     = '4',
        'comment'                    = 'Bronze layer: inventario farmaceutico raw. Schema-on-read. No transformaciones.'
    )
""")

print(f"Tabla creada o verificada: {TABLE_BRONZE}")

# Verificar propiedades
spark.sql(f"DESCRIBE DETAIL {TABLE_BRONZE}").select(
    "name", "numFiles", "sizeInBytes", "partitionColumns"
).show(truncate=False)


In [0]:
%sql
DESCRIBE DETAIL dbassociate.default.bronze_inventario_farmaceutico

In [0]:
%sql
DESCRIBE HISTORY dbassociate.default.bronze_inventario_farmaceutico

## Celda 6 — Crear la tabla de control ingestion_log

Esta tabla registra cada ejecución del pipeline de ingesta.  
Es la fuente de verdad para monitoreo operativo y SLAs.

> **Diseño deliberado:** Esta tabla usa `MERGE` en la escritura (celda 8)  
> para garantizar idempotencia. Si el pipeline falla y se reintenta con  
> el mismo `batch_id`, el log no genera filas duplicadas.


In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_LOG} (
        batch_id         STRING        NOT NULL,
        archivo          STRING,
        source_system    STRING,
        registros_leidos BIGINT,
        estado           STRING,
        inicio           TIMESTAMP,
        fin              TIMESTAMP,
        duracion_seg     DOUBLE,
        tabla_destino    STRING,
        log_timestamp    TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'false',
        'comment'                    = 'Log de control de ingestas Bronze. Idempotente por batch_id.'
    )
""")

print(f"Tabla de log creada o verificada: {TABLE_LOG}")


In [0]:
%sql
DESCRIBE DETAIL  dbassociate.default.bronze_ingestion_log

## Celda 7 — Función para registrar en el ingestion_log

Usa MERGE para garantizar idempotencia:  
si el mismo `batch_id + archivo` ya existe, actualiza en lugar de insertar duplicado.


In [0]:
def registrar_log(metricas: dict):
    """
    Registra las métricas de ingesta en bronze_ingestion_log de forma idempotente.
    Usa MERGE para evitar duplicados si el pipeline se reintenta.
    """
    from pyspark.sql.types import LongType
    
    df_log = spark.createDataFrame([{
        "batch_id":         metricas["batch_id"],
        "archivo":          metricas["archivo"],
        "source_system":    metricas["source_system"],
        "registros_leidos": metricas["registros_leidos"],
        "estado":           metricas["estado"],
        "inicio":           metricas["inicio"],
        "fin":              metricas["fin"],
        "duracion_seg":     float(metricas["duracion_seg"]),
        "tabla_destino":    metricas["tabla_destino"],
        "log_timestamp":    datetime.now(),
    }])
    
    df_log.createOrReplaceTempView("log_staging")
    
    spark.sql(f"""
        MERGE INTO {TABLE_LOG} AS target
        USING log_staging AS source
        ON target.batch_id = source.batch_id
           AND target.archivo = source.archivo
        WHEN MATCHED THEN
            UPDATE SET
                estado           = source.estado,
                registros_leidos = source.registros_leidos,
                fin              = source.fin,
                duracion_seg     = source.duracion_seg,
                log_timestamp    = source.log_timestamp
        WHEN NOT MATCHED THEN
            INSERT *
    """)
    
    print(f"Log registrado: batch_id={metricas['batch_id'][:8]}... | "
          f"archivo={metricas['archivo']} | "
          f"registros={metricas['registros_leidos']} | "
          f"estado={metricas['estado']} | "
          f"duracion={metricas['duracion_seg']:.2f}s")

print("Función registrar_log definida.")


## Celda 8 — Ejecutar ingesta del Batch 01

Primer lote de inventario: 20 productos de línea base.


In [0]:
ruta_batch01 = f"{VOLUME_PATH}/inventario_batch01.csv"
print(f"Iniciando ingesta: {ruta_batch01}")

metricas_01 = ingestar_batch_bronze(ruta_batch01, BATCH_ID)
registrar_log(metricas_01)

print(f"\nBatch 01 completado en {metricas_01['duracion_seg']:.2f} segundos.")


In [0]:
%sql
select count(1) from dbassociate.default.bronze_inventario_farmaceutico LIMIT 5

In [0]:
%sql
select * from dbassociate.default.bronze_ingestion_log

## Celda 9 — Ejecutar ingesta del Batch 02

Segundo lote: nuevos productos + actualizaciones de stock de productos existentes.  
Bronze no deduplica: ambas versiones quedan registradas con su timestamp de ingesta.  
La deduplicación es responsabilidad de Silver.


In [0]:
# Nota: batch02 contiene registros de P001, P005, P009, P011 que ya existen en batch01
# Bronze los acepta todos. El historial completo de recepciones queda registrado.
# Esto es correcto en Bronze: fidelidad al origen sobre cualquier transformación.
ruta_batch02 = f"{VOLUME_PATH}/inventario_batch02.csv"
print(f"Iniciando ingesta: {ruta_batch02}")

metricas_02 = ingestar_batch_bronze(ruta_batch02, BATCH_ID)
registrar_log(metricas_02)

print(f"\nBatch 02 completado en {metricas_02['duracion_seg']:.2f} segundos.")


In [0]:
%sql
select * from dbassociate.default.bronze_inventario_farmaceutico
where producto_id = 'P001'

## Celda 10 — Verificar contenido de la tabla Bronze

Exploramos lo que se ingestó: total de registros, distribución por partición  
y presencia de columnas técnicas.


In [0]:
print("=== CONTEO TOTAL EN BRONZE ===")
spark.sql(f"SELECT COUNT(*) AS total_registros FROM {TABLE_BRONZE}").show()

print("=== DISTRIBUCION POR PARTICION (ingestion_date) ===")
spark.sql(f"""
    SELECT 
        ingestion_date,
        source_file,
        COUNT(*) AS registros,
        COUNT(DISTINCT producto_id) AS productos_distintos
    FROM {TABLE_BRONZE}
    GROUP BY ingestion_date, source_file
    ORDER BY ingestion_date, source_file
""").show(truncate=False)

print("=== MUESTRA CON COLUMNAS TECNICAS ===")
spark.sql(f"""
    SELECT 
        producto_id, 
        nombre_producto,
        unidades_stock,
        ingestion_timestamp,
        source_file,
        batch_id,
        ingestion_date
    FROM {TABLE_BRONZE}
    LIMIT 5
""").show(truncate=False)


## Celda 11 — Verificar duplicados esperados en Bronze

Los registros P001, P005, P009, P011 aparecen en ambos batches.  
En Bronze esto es correcto: representa dos recepciones distintas del mismo producto.  
Mostrar el historial completo por producto_id.


In [0]:
print("=== PRODUCTOS QUE APARECEN EN MAS DE UN BATCH (comportamiento esperado en Bronze) ===")
spark.sql(f"""
    SELECT 
        producto_id,
        nombre_producto,
        unidades_stock,
        source_file,
        ingestion_timestamp
    FROM {TABLE_BRONZE}
    WHERE producto_id IN ('P001', 'P005', 'P009', 'P011')
    ORDER BY producto_id, ingestion_timestamp
""").show(truncate=False)

print("""
EXPLICACION:
- Bronze preserva TODOS los registros recibidos, incluyendo duplicados de negocio.
- P001 aparece 2 veces: una con stock 2400 (batch01) y otra con 2650 (batch02).
- Ambas son versiones validas del inventario en distintos momentos de carga.
- Silver sera responsable de resolver cual es el stock vigente usando la logica de negocio.
""")


## Celda 12 — Verificar el ingestion_log

El log de control registra todas las ejecuciones y es la fuente de verdad  
para monitoreo operativo.


In [0]:
print("=== INGESTION LOG ===")
spark.sql(f"""
    SELECT 
        batch_id,
        archivo,
        registros_leidos,
        estado,
        duracion_seg,
        inicio,
        fin
    FROM {TABLE_LOG}
    ORDER BY inicio
""").show(truncate=False)

print("\n=== RESUMEN DEL BATCH ACTUAL ===")
spark.sql(f"""
    SELECT 
        batch_id,
        SUM(registros_leidos) AS total_registros,
        COUNT(*) AS archivos_procesados,
        MIN(inicio) AS inicio_pipeline,
        MAX(fin) AS fin_pipeline,
        ROUND(SUM(duracion_seg), 2) AS duracion_total_seg
    FROM {TABLE_LOG}
    WHERE batch_id = '{BATCH_ID}'
    GROUP BY batch_id
""").show(truncate=False)


## Celda 13 — Explorar el transaction log de Delta

Delta Lake mantiene un transaction log en `_delta_log/`.  
Cada operación de escritura genera una entrada en el log.  
`DESCRIBE HISTORY` muestra el historial de operaciones sobre la tabla.


In [0]:
print("=== HISTORIAL DE OPERACIONES DELTA (transaction log) ===")
spark.sql(f"DESCRIBE HISTORY {TABLE_BRONZE}").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

print("\n=== DETALLE DE LA TABLA BRONZE ===")
spark.sql(f"DESCRIBE DETAIL {TABLE_BRONZE}").select(
    "name", "format", "numFiles", "sizeInBytes", "partitionColumns", "properties"
).show(truncate=False)


In [0]:
%sql
describe history dbassociate.default.bronze_inventario_farmaceutico

## Celda 14 — Ejecutar OPTIMIZE en la tabla Bronze

OPTIMIZE compacta los small files generados por las escrituras individuales.  
En una tabla Bronze de producción se programaría periódicamente (ej. cada noche).

> **Antipatrón:** ejecutar OPTIMIZE después de cada micro-batch en streaming.  
> El overhead de OPTIMIZE en batches pequeños supera el beneficio.  
> Programar OPTIMIZE como job independiente en ventana de mantenimiento.


In [0]:
print("=== ESTADO ANTES DE OPTIMIZE ===")
antes_detail = (
    spark.sql(f"DESCRIBE DETAIL {TABLE_BRONZE}")
    .select("numFiles", "sizeInBytes")
    .collect()[0]
)
print(f"Archivos antes: {antes_detail['numFiles']} | Tamaño: {antes_detail['sizeInBytes']} bytes")

print("\nEjecutando OPTIMIZE...")
spark.sql(f"OPTIMIZE {TABLE_BRONZE}")

print("\n=== ESTADO DESPUES DE OPTIMIZE ===")
despues_detail = (
    spark.sql(f"DESCRIBE DETAIL {TABLE_BRONZE}")
    .select("numFiles", "sizeInBytes")
    .collect()[0]
)
print(f"Archivos despues: {despues_detail['numFiles']} | Tamaño: {despues_detail['sizeInBytes']} bytes")

print("\nNota: en conjuntos de datos pequenos, OPTIMIZE puede no reducir el numero de archivos.")
print("El beneficio se observa con cientos de small files generados por muchos batches.")

## Celda 15 — Idempotencia: prueba de reintento

Simulamos que el pipeline falla y se vuelve a ejecutar con el mismo `batch_id`.  
Verificamos que el ingestion_log no genera duplicados gracias al MERGE.

> **Nota:** La tabla Bronze SÍ acumulará registros si se vuelve a ejecutar la ingesta  
> completa (celda 8 y 9) sin limpiar. En producción, la idempotencia de la escritura  
> a Bronze se controla con Auto Loader + checkpoint (sesión 03).  
> Aquí demostramos solo la idempotencia del LOG.


In [0]:
TABLE_LOG

In [0]:
%sql
SELECT * FROM dbassociate.default.bronze_ingestion_log


In [0]:
print("Simulando reintento del log con el mismo batch_id...")

# Mismo batch_id que el batch01 original
metricas_reintento = {
    "batch_id":         BATCH_ID,  # mismo ID
    "archivo":          "inventario_batch01.csv",
    "source_system":    SOURCE_SYSTEM,
    "registros_leidos": 20,
    "estado":           "COMPLETADO",
    "inicio":           datetime.now(),
    "fin":              datetime.now(),
    "duracion_seg":     0.1,
    "tabla_destino":    TABLE_BRONZE,
}

registrar_log(metricas_reintento)

print("\n=== LOG DESPUES DEL REINTENTO (debe seguir teniendo 2 filas, no 3) ===")
spark.sql(f"SELECT batch_id, archivo, estado, registros_leidos FROM {TABLE_LOG}").show(truncate=False)


In [0]:
%sql
SELECT * FROM dbassociate.default.bronze_ingestion_log

## Celda 16 — Leer tabla de referencia de almacenes

Los datos de referencia también se ingresan a Bronze antes de cualquier join.  
Silver usará esta tabla para enriquecer los datos de inventario.


In [0]:
schema_almacenes = StructType([
    StructField("almacen_id",             StringType(),  nullable=False),
    StructField("nombre_almacen",         StringType(),  nullable=True),
    StructField("ciudad",                 StringType(),  nullable=True),
    StructField("region",                 StringType(),  nullable=True),
    StructField("tipo_almacen",           StringType(),  nullable=True),
    StructField("temperatura_controlada", StringType(),  nullable=True),
    StructField("capacidad_m3",           IntegerType(), nullable=True),
])

TABLE_BRONZE_ALM = f"{CATALOG}.{SCHEMA}.bronze_almacenes_referencia"

df_almacenes = (
    spark.read
    .schema(schema_almacenes)
    .option("header", "true")
    .csv(f"{VOLUME_PATH}/almacenes_referencia.csv")
)

df_almacenes_bronze = df_almacenes.withColumns({
    "ingestion_timestamp": F.current_timestamp(),
    "source_file":         F.lit("almacenes_referencia.csv"),
    "source_system":       F.lit(SOURCE_SYSTEM),
    "batch_id":            F.lit(BATCH_ID),
    "ingestion_date":      F.to_date(F.current_timestamp()),
})

(
    df_almacenes_bronze.write
    .format("delta")
    .mode("overwrite")       # Tabla de referencia: overwrite total en cada carga
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_BRONZE_ALM)
)

print(f"Tabla de referencia Bronze creada: {TABLE_BRONZE_ALM}")
spark.sql(f"SELECT * FROM {TABLE_BRONZE_ALM}").show(truncate=False)


In [0]:
%sql
select * from dbassociate.default.bronze_almacenes_referencia

## Celda 17 — Time Travel: consultar versión anterior de Bronze

Una de las ventajas clave de Delta Lake en Bronze es el time travel.  
Podemos consultar el estado de la tabla antes del segundo batch.


In [0]:
print("=== HISTORIAL COMPLETO ===")
historial = spark.sql(f"DESCRIBE HISTORY {TABLE_BRONZE}").select(
    "version", "timestamp", "operation"
).collect()

for fila in historial:
    print(f"  Version {fila['version']} | {fila['timestamp']} | {fila['operation']}")

# Consultar la version inicial (solo batch01)
if len(historial) >= 2:
    version_batch01 = historial[-1]['version']  # la mas antigua
    print(f"\n=== CONTEO EN VERSION {version_batch01} (solo batch01) ===")
    spark.sql(f"""
        SELECT COUNT(*) AS registros_version_{version_batch01}
        FROM {TABLE_BRONZE} VERSION AS OF {version_batch01}
    """).show()
    
    print(f"=== CONTEO EN VERSION ACTUAL ===")
    spark.sql(f"SELECT COUNT(*) AS registros_actual FROM {TABLE_BRONZE}").show()
else:
    print("Solo hay una version disponible. Ejecutar ambos batches para ver time travel.")


In [0]:
%sql
describe history dbassociate.default.bronze_inventario_farmaceutico

In [0]:
%sql
SELECT count(1) FROM dbassociate.default.bronze_inventario_farmaceutico@v2

## Celda 18 — Limpieza del laboratorio

> Ejecutar esta celda solo al finalizar el laboratorio o para reiniciar desde cero.  
> En un entorno compartido, confirmar con el instructor antes de limpiar.


In [0]:
# Descomentar para ejecutar la limpieza
# ADVERTENCIA: elimina las tablas y todos sus datos

# spark.sql(f"DROP TABLE IF EXISTS {TABLE_BRONZE}")
# spark.sql(f"DROP TABLE IF EXISTS {TABLE_BRONZE_ALM}")
# spark.sql(f"DROP TABLE IF EXISTS {TABLE_LOG}")
# print("Tablas eliminadas.")

print("Celda de limpieza lista. Descomentar las lineas para ejecutar.")
print(f"Tablas que se eliminarian:")
print(f"  - {TABLE_BRONZE}")
print(f"  - {TABLE_BRONZE_ALM}")
print(f"  - {TABLE_LOG}")


## Resumen del laboratorio

| Concepto | Implementacion en este lab |
|---|---|
| Columnas tecnicas | ingestion_timestamp, source_file, source_system, batch_id, ingestion_date |
| Particionamiento | Por `ingestion_date` (fecha tecnica, no de negocio) |
| Formato | Delta Lake con `delta.enableChangeDataFeed = true` |
| Idempotencia del log | MERGE en ingestion_log por batch_id + archivo |
| Fidelidad de origen | Duplicados de negocio preservados en Bronze |
| Observabilidad | ingestion_log con metricas por ejecucion |
| Mantenimiento | OPTIMIZE ejecutado manualmente (en produccion: job programado) |
| Time travel | DESCRIBE HISTORY + VERSION AS OF |
